In [101]:
import json
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import silhouette_score, pairwise_distances
from sklearn.preprocessing import LabelEncoder

sys.path.insert(0, str(Path("..").resolve()))

ANALYSIS_DIR = Path("../data/analysis")
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = Path("../data/output")

CONDITION_LABELS = {
    "full":           "Full formulation",
    "no_formulation": "No formulation",
    "zero_shot":      "Zero-shot",
}
CONDITION_COLORS = {
    "full":           "#2196F3",
    "no_formulation": "#FF9800",
    "zero_shot":      "#4CAF50",
}
EVAL_MODELS = {
    "gpt-5.4", "gpt-5.4-mini", "gpt-4o-mini",
    "claude-sonnet-4-6", "claude-haiku-4-5",
    "gemini-2.5-flash",
    "deepseek-chat", "deepseek-reasoner",
    "qwen2.5-32b", "qwen3.6-35b",
}
print("Ready")

Ready


## 1. Load vignettes (cached)

In [102]:
DF_CACHE = ANALYSIS_DIR / "all_vignettes.parquet"

if DF_CACHE.exists():
    df = pd.read_parquet(DF_CACHE)
    print(f"Loaded {len(df)} cached vignettes")
else:
    from utils.text import strip_markdown
    groups = defaultdict(list)
    for d in OUTPUT_DIR.iterdir():
        if not d.is_dir() or d.name.startswith("craft_persona"):
            continue
        files = list(d.glob("experiment_*.json"))
        if not files:
            continue
        try:
            with open(files[0], encoding="utf-8") as f:
                cfg = json.load(f).get("config", {})
            model     = next(iter(cfg.get("models", {}).values()), None)
            condition = cfg.get("vignette_mode")
            if model and condition and model in EVAL_MODELS:
                groups[(model, condition)].append((d, len(files)))
        except Exception:
            continue
    best_dirs = {k: max(v, key=lambda x: x[1])[0] for k, v in groups.items()}
    print(f"Found {len(best_dirs)} (model, condition) groups")
    rows = []
    for (model, condition), dirpath in sorted(best_dirs.items()):
        for fp in sorted(dirpath.glob("experiment_*.json")):
            try:
                with open(fp, encoding="utf-8") as f:
                    data = json.load(f)
                text = strip_markdown(data.get("vignette", ""))
                if not text:
                    continue
                pid = data.get("persona_id") or data.get("source_persona_id")
                rows.append({"persona_id": pid, "model": model,
                             "condition": condition, "vignette": text,
                             "word_count": len(text.split())})
            except Exception:
                continue
    df = pd.DataFrame(rows)
    df.to_parquet(DF_CACHE, index=False)
    print(f"Saved -> {DF_CACHE}")

print(f"Total: {len(df)} vignettes, {df['persona_id'].nunique()} personas")
df.groupby(["condition", "model"]).size().unstack(fill_value=0)

Found 30 (model, condition) groups
Saved -> ..\data\analysis\all_vignettes.parquet
Total: 14981 vignettes, 500 personas


model,claude-haiku-4-5,claude-sonnet-4-6,deepseek-chat,deepseek-reasoner,gemini-2.5-flash,gpt-4o-mini,gpt-5.4,gpt-5.4-mini,qwen2.5-32b,qwen3.6-35b
condition,,,,,,,,,,
full,500,500,500,499,499,500,500,500,500,500
no_formulation,500,500,500,500,500,500,500,500,500,499
zero_shot,500,500,500,500,484,500,500,500,500,500


## 2. Load all-mpnet-base-v2 embeddings (cached)
768-d, L2-normalised. Delete `embeddings_mpnet.npy` to recompute (~10 min).

In [ ]:
from sentence_transformers import SentenceTransformer

EMB_CACHE = ANALYSIS_DIR / "embeddings_mpnet.npy"

if EMB_CACHE.exists():
    embeddings = np.load(EMB_CACHE)
    print(f"Loaded cached embeddings: {embeddings.shape}")
    assert len(embeddings) == len(df), (
        f"Cache mismatch: {len(embeddings)} vs {len(df)} rows. "
        f"Delete {EMB_CACHE} and rerun."
    )
else:
    print(f"Computing embeddings for {len(df)} vignettes (~10 min)...")
    enc = SentenceTransformer("all-mpnet-base-v2")
    embeddings = enc.encode(
        df["vignette"].tolist(),
        show_progress_bar=True,
        batch_size=32,
        normalize_embeddings=True,
    )
    np.save(EMB_CACHE, embeddings)
    print(f"Saved -> {EMB_CACHE}  shape={embeddings.shape}")

Computing embeddings for 14981 vignettes (~10 min)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/469 [00:00<?, ?it/s]

## 3. UMAP — per model × condition (cached)

In [ ]:
import umap

UMAP_CACHE = ANALYSIS_DIR / "umap_coords_per_condition_mpnet.npz"
N_NEIGHBORS = 5

if UMAP_CACHE.exists():
    _c = np.load(UMAP_CACHE)
    umap_coords = {k: _c[k] for k in _c.files}
    print("Loaded cached UMAP coords:", list(umap_coords.keys()))
else:
    umap_coords = {}
    for cond in CONDITION_LABELS:
        idx  = df[df["condition"] == cond].index.tolist()
        embs = embeddings[idx]
        print(f"{CONDITION_LABELS[cond]}: {len(idx)} vignettes...")
        coords = umap.UMAP(n_components=2, n_neighbors=N_NEIGHBORS,
                           min_dist=0.1, random_state=42).fit_transform(embs)
        umap_coords[cond] = coords
    np.savez(UMAP_CACHE, **umap_coords)
    print(f"Saved -> {UMAP_CACHE}")

models     = sorted(df["model"].unique())
conditions = list(CONDITION_LABELS.keys())
nrows, ncols = len(models), len(conditions)

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5, nrows * 4))
for r, model in enumerate(models):
    for c, cond in enumerate(conditions):
        ax = axes[r, c]
        idx        = df[df["condition"] == cond].index
        cond_df    = df.loc[idx].reset_index(drop=True)
        coords     = umap_coords[cond]
        model_mask = (cond_df["model"] == model).values
        ax.scatter(coords[model_mask, 0], coords[model_mask, 1],
                   c=CONDITION_COLORS[cond], s=12, alpha=0.85, edgecolors="none")
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(True); spine.set_linewidth(0.6); spine.set_color("#cccccc")
        if r == 0:
            ax.set_title(CONDITION_LABELS[cond], fontsize=12, fontweight="bold", pad=8)
        if c == 0:
            ax.set_title(model, loc="left", fontsize=10, fontweight="bold", pad=8)

plt.suptitle("Vignette Semantic Space — all-mpnet-base-v2 + UMAP",
             fontsize=14, y=1.005)
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / "umap_per_model_grid_mpnet.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved -> {ANALYSIS_DIR}/umap_per_model_grid_mpnet.png")

## 4. t-SNE — per model × condition (cached)

In [ ]:
from sklearn.manifold import TSNE

TSNE_CACHE = ANALYSIS_DIR / "tsne_coords_per_condition_mpnet.npz"

if TSNE_CACHE.exists():
    _c = np.load(TSNE_CACHE)
    tsne_coords = {k: _c[k] for k in _c.files}
    print("Loaded cached t-SNE coords:", list(tsne_coords.keys()))
else:
    tsne_coords = {}
    for cond in CONDITION_LABELS:
        idx  = df[df["condition"] == cond].index.tolist()
        embs = embeddings[idx]
        print(f"{CONDITION_LABELS[cond]}: running t-SNE on {len(idx)} vignettes...")
        coords = TSNE(n_components=2, perplexity=30, learning_rate="auto",
                      init="pca", random_state=42, n_jobs=-1).fit_transform(embs)
        tsne_coords[cond] = coords
        print(f"  done {coords.shape}")
    np.savez(TSNE_CACHE, **tsne_coords)
    print(f"Saved -> {TSNE_CACHE}")

models     = sorted(df["model"].unique())
conditions = list(CONDITION_LABELS.keys())
nrows, ncols = len(models), len(conditions)

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5, nrows * 4))
for r, model in enumerate(models):
    for c, cond in enumerate(conditions):
        ax = axes[r, c]
        idx        = df[df["condition"] == cond].index
        cond_df    = df.loc[idx].reset_index(drop=True)
        coords     = tsne_coords[cond]
        model_mask = (cond_df["model"] == model).values
        ax.scatter(coords[model_mask, 0], coords[model_mask, 1],
                   c=CONDITION_COLORS[cond], s=12, alpha=0.85, edgecolors="none")
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(True); spine.set_linewidth(0.6); spine.set_color("#cccccc")
        if r == 0:
            ax.set_title(CONDITION_LABELS[cond], fontsize=12, fontweight="bold", pad=8)
        if c == 0:
            ax.set_title(model, loc="left", fontsize=10, fontweight="bold", pad=8)

plt.suptitle("Vignette Semantic Space — all-mpnet-base-v2 + t-SNE",
             fontsize=14, y=1.005)
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / "tsne_per_model_grid_mpnet.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved -> {ANALYSIS_DIR}/tsne_per_model_grid_mpnet.png")

## 5. Mean pairwise cosine distance — per model × condition

In [ ]:
rows = []
for model in sorted(df["model"].unique()):
    for cond, label in CONDITION_LABELS.items():
        idx  = df[(df["model"] == model) & (df["condition"] == cond)].index
        embs = embeddings[idx]
        dists = pairwise_distances(embs, metric="cosine")
        upper = dists[np.triu_indices_from(dists, k=1)]
        rows.append({"model": model, "condition": label,
                     "mean_dist": round(float(upper.mean()), 4)})

cosine_df = pd.DataFrame(rows)
pivot = cosine_df.pivot(index="model", columns="condition", values="mean_dist")
pivot = pivot[[CONDITION_LABELS[c] for c in CONDITION_LABELS]]
pivot["mean"] = pivot.mean(axis=1)
pivot = pivot.sort_values("mean", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [3, 1]})

data = pivot[[CONDITION_LABELS[c] for c in CONDITION_LABELS]]
sns.heatmap(data, ax=axes[0], annot=True, fmt=".4f", cmap="YlOrRd",
            linewidths=0.5, cbar_kws={"label": "Mean pairwise cosine dist"})
axes[0].set_title("Semantic diversity per model × condition", fontsize=12)
axes[0].set_xlabel(""); axes[0].set_ylabel("")
axes[0].tick_params(axis="x", rotation=15)

axes[1].barh(pivot.index, pivot["mean"], color="#5C6BC0", alpha=0.85)
axes[1].set_xlabel("Mean across conditions", fontsize=10)
axes[1].set_title("Overall diversity", fontsize=12)
axes[1].spines[["top", "right"]].set_visible(False)
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig(ANALYSIS_DIR / "diversity_per_model_heatmap_mpnet.png", dpi=150, bbox_inches="tight")
plt.show()
pivot

## 6. Silhouette score — per condition
- **S_model**: do models produce distinct clusters within a condition?  
- **S_persona**: does persona drive semantic content?

Near +1 → label predicts clusters. Near 0 → interleaved.

In [ ]:
sil_rows = []
for cond, label in CONDITION_LABELS.items():
    idx      = df[df["condition"] == cond].index
    embs     = embeddings[idx]
    labels_m = LabelEncoder().fit_transform(df.loc[idx, "model"])
    labels_p = LabelEncoder().fit_transform(df.loc[idx, "persona_id"])
    s_m = silhouette_score(embs, labels_m, metric="cosine")
    s_p = silhouette_score(embs, labels_p, metric="cosine")
    sil_rows.append({"Condition": label,
                     "Silhouette (model)": round(s_m, 4),
                     "Silhouette (persona)": round(s_p, 4)})
    print(f"{label:<22}  S_model={s_m:.4f}  S_persona={s_p:.4f}")

sil_df = pd.DataFrame(sil_rows).set_index("Condition")

fig, ax = plt.subplots(figsize=(7, 3))
x = np.arange(len(sil_df))
w = 0.35
ax.bar(x - w/2, sil_df["Silhouette (model)"],   width=w, label="by model",   color="#5C6BC0", alpha=0.85)
ax.bar(x + w/2, sil_df["Silhouette (persona)"], width=w, label="by persona", color="#EF5350", alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(sil_df.index, rotation=10)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_ylabel("Silhouette score")
ax.set_title("Silhouette scores per condition")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / "silhouette_mpnet.png", dpi=150, bbox_inches="tight")
plt.show()
sil_df

## 7. Vendi Score — per model × condition
Effective number of distinct vignettes. Rewards uniform spread, penalises clustering.  
Higher = more diverse.

In [ ]:
def vendi_score(embs):
    n = len(embs)
    K = embs @ embs.T
    K = K / n
    eigvals = np.linalg.eigvalsh(K)
    eigvals = eigvals[eigvals > 1e-10]
    eigvals /= eigvals.sum()
    return float(np.exp(-np.sum(eigvals * np.log(eigvals))))

rows = []
for model in sorted(df["model"].unique()):
    for cond, label in CONDITION_LABELS.items():
        idx  = df[(df["model"] == model) & (df["condition"] == cond)].index
        rows.append({"model": model, "condition": label,
                     "vendi_score": round(vendi_score(embeddings[idx]), 1)})

vendi_df = pd.DataFrame(rows)
pivot_v  = vendi_df.pivot(index="model", columns="condition", values="vendi_score")
pivot_v  = pivot_v[[CONDITION_LABELS[c] for c in CONDITION_LABELS]]
pivot_v["mean"] = pivot_v.mean(axis=1)
pivot_v  = pivot_v.sort_values("mean", ascending=False)
pivot_v.to_csv(ANALYSIS_DIR / "vendi_per_model_mpnet.csv")

fig, ax = plt.subplots(figsize=(7, 5))
data = pivot_v[[CONDITION_LABELS[c] for c in CONDITION_LABELS]]
sns.heatmap(data, ax=ax, annot=True, fmt=".0f", cmap="YlOrRd",
            linewidths=0.5, cbar_kws={"label": "Vendi Score"})
ax.set_xlabel(""); ax.set_ylabel("")
ax.set_title("Vendi Score per model × condition", fontsize=12)
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / "vendi_heatmap_mpnet.png", dpi=150, bbox_inches="tight")
plt.show()
pivot_v